# Домашнее задание: user_data.xlsx

**Таблицы:** users_behaviour, users_info, users_acquisition

1. Применить **describe** к двум таблицам (например users_behaviour и users_acquisition)  
2. Распределение **rating**: гистограмма и подбор количества корзин  
3. **Ящик с усами** (размах); **скрипичная диаграмма** по rating, разбивка по ОС (два скрипплата: iOS и Android)  
4. Сумма по полям adult_flag, higher_education_flag, employed_flag, bank_account_flag, married_flag → **столбчатая диаграмма**, сортировка по убыванию, вывод о самом/наименее популярном признаке  
5. **Столбчатая диаграмма**: количество пользователей по датам привлечения (без дублей), сравнение дат  
6. Доля каждой ОС в объёме пользователей → **круговая диаграмма** с подписями значений  

**Дополнительно:**  
7. **Тепловая карта**: когортный анализ по **среднему чеку** по когортам  
8. **Воронка** по количеству user_id с процентом от предыдущего этапа

In [ ]:
# Загрузка данных (файл user_data.xlsx — в папке с ноутбуком)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

xlsx = "user_data.xlsx"
behaviour = pd.read_excel(xlsx, sheet_name="users_behaviour")
info = pd.read_excel(xlsx, sheet_name="users_info")
acquisition = pd.read_excel(xlsx, sheet_name="users_acquisition")
print("users_behaviour:", behaviour.shape)
print("users_info:", info.shape)
print("users_acquisition:", acquisition.shape)

In [ ]:
# Задание 1: describe для двух таблиц
print("=== users_behaviour ===")
display(behaviour.describe())
print("\n=== users_acquisition ===")
display(acquisition.describe())

In [ ]:
# Задание 2: распределение rating — гистограмма и подбор корзин
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
info["rating"].hist(ax=axes[0], bins=20, edgecolor="black", alpha=0.7)
axes[0].set_title("rating, 20 корзин")
axes[0].set_xlabel("rating")
# Правило Стерджеса: k ≈ 1 + log2(n)
import numpy as np
n = len(info["rating"].dropna())
k = int(1 + np.log2(n)) if n > 0 else 10
info["rating"].hist(ax=axes[1], bins=k, edgecolor="black", alpha=0.7)
axes[1].set_title(f"rating, по Стерджесу ({k} корзин)")
axes[1].set_xlabel("rating")
plt.tight_layout()
plt.show()
print("Количество корзин по правилу Стерджеса:", k)

In [ ]:
# Задание 3: ящик с усами (размах) и скрипичные диаграммы по ОС (iOS и Android)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# Ящик с усами
info.boxplot(column="rating", ax=axes[0])
axes[0].set_title("Ящик с усами: rating")
# Два скрипплата: iOS и Android
for i, os_name in enumerate(["iOS", "Android"]):
    data = info[info["operating_system"] == os_name]["rating"]
    axes[i + 1].violinplot([data.dropna().values], positions=[0])
    axes[i + 1].set_title(f"Скриплот: {os_name}")
    axes[i + 1].set_ylabel("rating")
plt.tight_layout()
plt.show()

In [ ]:
# Задание 4: сумма по флагам → столбчатая диаграмма, сортировка по убыванию
flags = ["adult_flag", "higher_education_flag", "employed_flag", "bank_account_flag", "married_flag"]
labels = ["Взрослый", "Высшее образование", "Трудоустроен", "Банк. счёт", "В браке"]
sums = info[flags].sum()
sums.index = labels
sums = sums.sort_values(ascending=False)
sums.plot(kind="bar", figsize=(8, 4), edgecolor="black")
plt.title("Сумма по признакам")
plt.ylabel("Количество")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print("Самый популярный:", sums.idxmax())
print("Наименее популярный:", sums.idxmin())

In [ ]:
# Задание 5: количество пользователей по датам привлечения (без дублей)
users_by_date = behaviour.groupby("acquisition_date")["user_id"].nunique()
users_by_date.plot(kind="bar", figsize=(12, 4), edgecolor="black")
plt.title("Количество уникальных пользователей по дате привлечения")
plt.xlabel("Дата привлечения")
plt.ylabel("Количество user_id")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Задание 6: доля ОС — круговая диаграмма с подписями
os_counts = info["operating_system"].value_counts()
plt.figure(figsize=(6, 6))
wedges, texts, autotexts = plt.pie(os_counts, labels=os_counts.index, autopct="%1.1f%%", startangle=90)
for t in autotexts:
    t.set_fontsize(12)
plt.title("Доля операционных систем среди пользователей")
plt.show()

In [ ]:
# Доп. задание 7: тепловая карта — когортный анализ по среднему чеку
# Когорта = дата привлечения (acquisition_date), период = day
cohort = behaviour.pivot_table(
    index="acquisition_date",
    columns="day",
    values="purchase_amount",
    aggfunc="mean"
)
plt.figure(figsize=(12, 6))
sns.heatmap(cohort, annot=False, fmt=".0f", cmap="YlOrRd")
plt.title("Когортный анализ: средний чек по когорте и дню")
plt.xlabel("День")
plt.ylabel("Дата привлечения (когорта)")
plt.tight_layout()
plt.show()

In [ ]:
# Доп. задание 8: воронка по user_id с процентом от предыдущего этапа
order = ["Saw Ad", "Visited Store", "Downloaded App", "Registered", "status"]
funnel = acquisition.groupby("stage")["user_id"].nunique()
funnel = funnel.reindex([s for s in order if s in funnel.index]).dropna()
prev = funnel.shift(1, fill_value=funnel.iloc[0])
pct_prev = (funnel / prev * 100).round(1)
pct_prev.iloc[0] = 100.0
df_funnel = pd.DataFrame({"user_id": funnel, "% от предыдущего": pct_prev})
print(df_funnel)
df_funnel["user_id"].plot(kind="bar", figsize=(8, 4))
plt.title("Воронка: количество user_id по этапам")
plt.ylabel("Количество user_id")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()